# LCA Gasification Neural Network Regression
This notebook trains a Keras neural network model to predict H₂ and CO yield from gasification process parameters.

In [3]:
%pip install pandas numpy matplotlib scikit-learn tensorflow


ERROR: Could not find a version that satisfies the requirement tensorflow (from versions: none)
ERROR: No matching distribution found for tensorflow
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
# Load your dataset
df = pd.read_csv("lca_dataset_sample.csv")

# Define input features and output targets
features = ["temperature", "reaction_time", "pressure", "steam_ratio", "feedstock", "gasifier_type"]
targets = ["H2_yield", "CO_yield"]

X = df[features]
y = df[targets]

In [ ]:
# Preprocessing: Scale numeric + one-hot encode categorical
numeric_features = ["temperature", "reaction_time", "pressure", "steam_ratio"]
categorical_features = ["feedstock", "gasifier_type"]

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(sparse=False), categorical_features)
])

X_processed = preprocessor.fit_transform(X)

In [ ]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X_processed, y, test_size=0.2, random_state=42)

In [ ]:
# Define neural network model
model = Sequential()
model.add(Dense(128, input_dim=X_train.shape[1], activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(64, activation='relu'))
model.add(Dense(32, activation='relu'))
model.add(Dense(2))  # 2 outputs: H2 and CO yield

model.compile(loss='mse', optimizer='adam', metrics=['mae'])

In [ ]:
# Train the model
history = model.fit(X_train, y_train, validation_split=0.2, epochs=150, batch_size=8, verbose=1)

In [ ]:
# Predict and evaluate
y_pred = model.predict(X_test)
print("R² H2:", r2_score(y_test["H2_yield"], y_pred[:, 0]))
print("R² CO:", r2_score(y_test["CO_yield"], y_pred[:, 1]))

In [ ]:
# Plot actual vs predicted
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.scatter(y_test["H2_yield"], y_pred[:, 0], alpha=0.7)
plt.title("Actual vs Predicted H2 Yield")
plt.xlabel("Actual"); plt.ylabel("Predicted")

plt.subplot(1, 2, 2)
plt.scatter(y_test["CO_yield"], y_pred[:, 1], alpha=0.7)
plt.title("Actual vs Predicted CO Yield")
plt.xlabel("Actual"); plt.ylabel("Predicted")
plt.tight_layout()
plt.show()